In [1]:
# ============================================================
# CELL 0 – ENVIRONMENT SETUP (Colab + Local)
# ============================================================
import os, sys

# 1. Detect if running on Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    PROJECT_ROOT = '/content/hate-speech-detection'
    if not os.path.exists(PROJECT_ROOT):
        os.system('git clone https://github.com/thong7d/hate-speech-detection.git ' + PROJECT_ROOT)
else:
    # Local-first logic: detect root even if running from root folder
    cwd = os.getcwd()
    if os.path.basename(cwd) == 'notebooks':
        PROJECT_ROOT = os.path.abspath(os.path.join(cwd, '..'))
    else:
        PROJECT_ROOT = cwd

# 2. Add src/ to Python path
SRC_PATH = os.path.join(PROJECT_ROOT, 'src')
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print(f'✅ Environment ready. PROJECT_ROOT: {PROJECT_ROOT}')
REPO_PATH = PROJECT_ROOT


✅ Environment ready. PROJECT_ROOT: /content/hate-speech-detection


In [2]:
# ============================================================
# CELL 1 — GPU VERIFICATION & MEMORY UTILITIES
# ============================================================
# Input:  None
# Output: Confirms T4 GPU is active, defines memory helper functions
# ============================================================
import torch
import gc


def print_gpu_memory(tag: str = ""):
    """
    Print current GPU VRAM usage at a named checkpoint.
    Call this before and after every major operation.
    """
    if not torch.cuda.is_available():
        print(f"[GPU {tag}] ⚠️  No CUDA GPU detected.")
        return
    allocated = torch.cuda.memory_allocated() / 1024 ** 3
    reserved  = torch.cuda.memory_reserved()  / 1024 ** 3
    total     = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    free      = total - reserved
    print(
        f"[GPU | {tag}] "
        f"Allocated: {allocated:.2f}GB | "
        f"Reserved: {reserved:.2f}GB | "
        f"Free: {free:.2f}GB / {total:.2f}GB"
    )


def clean_memory():
    """
    Force Python garbage collection and clear GPU cache.
    Call this before any large allocation (model load, dataset build, training).
    """
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("🧹 Memory cleaned (gc.collect + cuda.empty_cache).")


# --- GPU assertion ---
assert torch.cuda.is_available(), (
    "❌ CUDA GPU not found!\n"
    "   Fix: Runtime → Change runtime type → T4 GPU → Save → Reconnect."
)

gpu_name = torch.cuda.get_device_name(0)
print(f"✅ GPU detected : {gpu_name}")
print(f"   torch version: {torch.__version__}")
print_gpu_memory("initial")


✅ GPU detected : Tesla T4
   torch version: 2.10.0+cu128
[GPU | initial] Allocated: 0.00GB | Reserved: 0.00GB | Free: 14.56GB / 14.56GB


In [3]:
# ============================================================
# CELL 2 — ROBUST DATA PIPELINE & CONFIG LOADING
# ============================================================
import os
import re
import json
import pandas as pd
import numpy as np
from omegaconf import OmegaConf
from sklearn.utils.class_weight import compute_class_weight

def resolve_path(template_path) -> str:
    """
    Resolve the {project_root} placeholder used in paths.yaml.
    Handles nested DictConfig robustness.
    """
    if isinstance(template_path, str):
        # Thay thế {project_root} và {drive_root} bằng REPO_PATH trên Colab
        return template_path.replace("{project_root}", REPO_PATH).replace("{drive_root}", REPO_PATH)

    if hasattr(template_path, "values"):
        for v in template_path.values():
            return resolve_path(v)
    return str(template_path)

# --- 1. Load config ---
cfg = OmegaConf.load(f"{REPO_PATH}/configs/paths.yaml")
print("✅ Config loaded from paths.yaml")

train_path = resolve_path(cfg.data.train_processed)
val_path   = resolve_path(cfg.data.val_processed)
test_path  = resolve_path(cfg.data.test_processed)
weights_path = resolve_path(cfg.results.class_weights)

# --- 3. Load Parquet Files ---
print(f"\n📥 Loading train : {train_path}")
print(f"📥 Loading val   : {val_path}")
print(f"📥 Loading test  : {test_path}")

train_df = pd.read_parquet(train_path)
val_df   = pd.read_parquet(val_path)
test_df  = pd.read_parquet(test_path)

# --- 4. Schema Validation ---
REQUIRED_COLUMNS = {"text", "label_id"}
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    missing = REQUIRED_COLUMNS - set(df.columns)
    assert not missing, (
        f"❌ '{name}' parquet is missing columns: {missing}\n"
        f"   Found columns: {list(df.columns)}"
    )

print(f"\n✅ Data loaded successfully:")
print(f"   Train : {len(train_df):,} rows | Label dist: {train_df['label_id'].value_counts().to_dict()}")
print(f"   Val   : {len(val_df):,} rows   | Label dist: {val_df['label_id'].value_counts().to_dict()}")
print(f"   Test  : {len(test_df):,} rows  | Label dist: {test_df['label_id'].value_counts().to_dict()}")

# --- 5. Load Class Weights ---
with open(weights_path, "r") as f:
    class_weights_raw = json.load(f)

if isinstance(class_weights_raw, list):
    class_weights = [float(w) for w in class_weights_raw]
elif isinstance(class_weights_raw, dict):
    class_weights = [float(class_weights_raw[str(i)]) for i in range(3)]
else:
    raise ValueError(f"Unexpected class_weights format: {type(class_weights_raw)}")

assert len(class_weights) == 3, f"❌ Expected 3 class weights, got {len(class_weights)}"
print(f"\n✅ Class weights: [CLEAN={class_weights[0]:.4f}, OFFENSIVE={class_weights[1]:.4f}, HATE={class_weights[2]:.4f}]")

✅ Config loaded from paths.yaml

📥 Loading train : /content/hate-speech-detection/data/processed/train.parquet
📥 Loading val   : /content/hate-speech-detection/data/processed/dev.parquet
📥 Loading test  : /content/hate-speech-detection/data/processed/test.parquet


FileNotFoundError: [Errno 2] No such file or directory: '/content/hate-speech-detection/data/processed/train.parquet'

In [ ]:
# ============================================================
# CELL 3 — TOKENIZER & DATASET CREATION
# ============================================================
# Input:  train_df, val_df, test_df from Cell 2
# Output: tokenizer (XLM-RoBERTa),
#         train_dataset, val_dataset, test_dataset (ViHSDDataset)
# Note:   Tokenizer is downloaded to /tmp/hf_cache (NOT Drive)
# ============================================================
from transformers import AutoTokenizer
from data.dataset import ViHSDDataset   # src/data/dataset.py

MODEL_NAME = "vinai/phobert-base-v2"
MAX_LENGTH = int(cfg.training.max_length)  # 128

clean_memory()
print(f"Loading tokenizer: '{MODEL_NAME}' → /tmp/hf_cache\n")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# --- Build datasets ---
# Uses ViHSDDataset from src/data/dataset.py
# Each __getitem__ returns: input_ids, attention_mask, labels (torch tensors)
train_dataset = ViHSDDataset(
    texts=train_df["text"].tolist(),
    labels=train_df["label_id"].tolist(),
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)
val_dataset = ViHSDDataset(
    texts=val_df["text"].tolist(),
    labels=val_df["label_id"].tolist(),
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)
test_dataset = ViHSDDataset(
    texts=test_df["text"].tolist(),
    labels=test_df["label_id"].tolist(),
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

# --- Sanity check: inspect one sample ---
sample = train_dataset[0]
assert set(sample.keys()) == {"input_ids", "attention_mask", "labels"}, (
    f"❌ Unexpected sample keys: {set(sample.keys())}"
)
assert sample["input_ids"].shape == (MAX_LENGTH,), (
    f"❌ input_ids shape mismatch: {sample['input_ids'].shape}"
)

print(f"✅ Tokenizer loaded: '{MODEL_NAME}'")
print(f"\n✅ Datasets created:")
print(f"   train_dataset : {len(train_dataset):,} samples")
print(f"   val_dataset   : {len(val_dataset):,} samples")
print(f"   test_dataset  : {len(test_dataset):,} samples")
print(f"\n   Sample[0] keys  : {list(sample.keys())}")
print(f"   input_ids shape : {sample['input_ids'].shape} (max_length={MAX_LENGTH})")
print(f"   label           : {sample['labels'].item()}")

print_gpu_memory("after_tokenizer_load")


NameError: name 'cfg' is not defined

In [ ]:
# ============================================================
# CELL 4 — MODEL INITIALIZATION
# ============================================================
# Input:  MODEL_NAME from Cell 3
# Output: model (XLM-RoBERTa for Sequence Classification)
# ============================================================
from transformers import AutoModelForSequenceClassification

print(f"Loading model: '{MODEL_NAME}'...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
)

print_gpu_memory("after_model_load")


In [ ]:
from transformers import Trainer

class WeightedTrainer(Trainer):
    """
    HuggingFace Trainer subclass with class-weighted loss and discriminative learning rates.
    """

    def __init__(self, class_weights: list, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Register as a plain tensor; move to correct device in compute_loss
        self._class_weights = torch.tensor(class_weights, dtype=torch.float32)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # Move weights to the same device as logits (GPU)
        weights = self._class_weights.to(logits.device)
        loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

    def create_optimizer(self):
        """
        Setup discriminative learning rates: backbone (lower LR) vs classification head (higher LR).
        """
        if self.optimizer is None:
            # Identify parameter groups
            # Note: PhoBERT/XLM-R both use .roberta for backbone and .classifier for head
            backbone_params = []
            head_params = []
            for name, param in self.model.named_parameters():
                if "classifier" in name:
                    head_params.append(param)
                else:
                    backbone_params.append(param)

            optimizer_grouped_parameters = [
                {"params": backbone_params, "lr": self.args.learning_rate},
                {"params": head_params, "lr": self.args.learning_rate * 5},
            ]

            self.optimizer = torch.optim.AdamW(
                optimizer_grouped_parameters,
                lr=self.args.learning_rate,
                eps=self.args.adam_epsilon,
                weight_decay=self.args.weight_decay,
            )
        return self.optimizer


In [ ]:
# ============================================================
# CELL 5 — TRAINING ARGUMENTS & CHECKPOINT RESUME DETECTION
# ============================================================
# Input:  cfg (OmegaConf), DRIVE_ROOT
# Output: training_args (TrainingArguments ready to pass to Trainer)
#         resume_checkpoint (str path or None)
#         CHECKPOINTS_DIR (Drive path where checkpoints are saved)
# ============================================================
import glob
import os
from transformers import TrainingArguments, EarlyStoppingCallback

# --- Checkpoint directory (on Drive, persists across sessions) ---
CHECKPOINTS_DIR = resolve_path(cfg.models.checkpoints_dir)
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
print(f"Checkpoints directory: {CHECKPOINTS_DIR}")

# ===========================================================
# AUTO-RESUME FUNCTION
# Finds the most recent VALID checkpoint to resume from.
# A "valid" checkpoint must have trainer_state.json (written
# only after a COMPLETE epoch save, not a mid-epoch crash).
# ===========================================================
def find_latest_checkpoint(checkpoints_dir: str):
    """
    Scan the checkpoint directory and return the path to the most
    recent valid checkpoint, or None if none exist.

    A valid checkpoint directory contains:
      - trainer_state.json  (proof that the epoch completed)
      - pytorch_model.bin OR model.safetensors (actual weights)

    Args:
        checkpoints_dir: Path to the directory containing checkpoint-N/ folders.

    Returns:
        str path to the latest valid checkpoint, or None.
    """
    # Find all checkpoint-* subdirectories
    pattern = os.path.join(checkpoints_dir, "checkpoint-*")
    candidates = sorted(
        glob.glob(pattern),
        key=lambda p: int(p.split("-")[-1]) if p.split("-")[-1].isdigit() else 0,
    )

    if not candidates:
        print("   No checkpoint folders found.")
        return None

    print(f"   Found {len(candidates)} checkpoint folder(s):")
    for c in candidates:
        print(f"     {os.path.basename(c)}")

    # Walk backwards to find the most recent VALID checkpoint
    for checkpoint_path in reversed(candidates):
        has_state = os.path.exists(
            os.path.join(checkpoint_path, "trainer_state.json")
        )
        has_weights = (
            os.path.exists(os.path.join(checkpoint_path, "pytorch_model.bin"))
            or os.path.exists(os.path.join(checkpoint_path, "model.safetensors"))
        )

        if has_state and has_weights:
            return checkpoint_path
        else:
            missing = []
            if not has_state:
                missing.append("trainer_state.json")
            if not has_weights:
                missing.append("weights (pytorch_model.bin / model.safetensors)")
            print(f"   ⚠️  Skipping {os.path.basename(checkpoint_path)} — missing: {missing}")

    print("   No VALID checkpoint found. Will train from scratch.")
    return None


# --- Detect checkpoint ---
print("\n🔍 Scanning for resume checkpoint...")
resume_checkpoint = find_latest_checkpoint(CHECKPOINTS_DIR)

if resume_checkpoint:
    print(f"\n🔄 RESUME MODE: Will resume from → {os.path.basename(resume_checkpoint)}")
else:
    print(f"\n🆕 FRESH START: Will train from scratch (no valid checkpoint found).")

# ===========================================================
# TRAINING ARGUMENTS
# All values sourced from paths.yaml — no hardcoding here.
# Key Colab-safety settings highlighted with comments.
# ===========================================================
training_args = TrainingArguments(
    output_dir=CHECKPOINTS_DIR,

    # --- Training schedule ---
    num_train_epochs=cfg.training.num_epochs,                       # 10
    per_device_train_batch_size=16,            # 16
    per_device_eval_batch_size=cfg.training.eval_batch_size,        # 32
    gradient_accumulation_steps=4,
    # --- Optimizer ---
    learning_rate=cfg.training.learning_rate,                       # 2e-5
    warmup_ratio=cfg.training.warmup_ratio,                         # 0.1
    weight_decay=cfg.training.weight_decay,                         # 0.01

    # --- Mixed precision (FP16): REQUIRED to fit on T4 ---
    fp16=bool(cfg.training.fp16),                                   # True

    # --- Evaluation & checkpointing ---
    eval_strategy="epoch",
    save_strategy="epoch",
    push_to_hub=True,
    hub_strategy="checkpoint",
    hub_model_id=str(cfg.hf_model_id),
    save_total_limit=int(cfg.training.save_total_limit),            # 2
    load_best_model_at_end=bool(cfg.training.load_best_model_at_end),  # True
    metric_for_best_model=str(cfg.training.metric_for_best_model),     # "eval_f1_macro"
    greater_is_better=True,

    # --- Logging ---
    logging_dir=f"{CHECKPOINTS_DIR}/logs",
    logging_steps=50,                  # Print training loss every 50 steps
    report_to="none",                  # Disable W&B / TensorBoard (conflicts on Colab)

    # --- CRITICAL Colab safety settings ---
    dataloader_num_workers=int(cfg.training.dataloader_num_workers),  # 0 (MUST be 0)
    dataloader_pin_memory=False,        # Disable pinning (unnecessary + uses RAM)

    # --- Gradient checkpointing compatibility ---
    # This suppresses the DDP warning about unused parameters
    # when gradient_checkpointing is enabled.
    ddp_find_unused_parameters=False,

    # --- Disable safetensors for compatibility with older trainer versions ---
    # Comment this out if you see "safetensors not installed" warnings
    # save_safetensors=False,
)

# --- Summary printout ---
print("\n" + "=" * 55)
print("✅ TrainingArguments configured:")
print(f"   Output dir          : {training_args.output_dir}")
print(f"   Num epochs          : {training_args.num_train_epochs}")
print(f"   Train batch size    : {training_args.per_device_train_batch_size}")
print(f"   Eval batch size     : {training_args.per_device_eval_batch_size}")
print(f"   Learning rate       : {training_args.learning_rate}")
print(f"   FP16                : {training_args.fp16}  ← MUST be True")
print(f"   num_workers         : {training_args.dataloader_num_workers}  ← MUST be 0")
print(f"   eval_strategy       : {training_args.eval_strategy}")
print(f"   save_total_limit    : {training_args.save_total_limit}")
print(f"   metric_for_best     : {training_args.metric_for_best_model}")
print(f"   Resume checkpoint   : {os.path.basename(resume_checkpoint) if resume_checkpoint else 'None (fresh start)'}")
print("=" * 55)
print_gpu_memory("cell5_end")


Checkpoints directory: /content/hate-speech-detection/models/xlmr_vihsd/checkpoints

🔍 Scanning for resume checkpoint...
   No checkpoint folders found.

🆕 FRESH START: Will train from scratch (no valid checkpoint found).


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



✅ TrainingArguments configured:
   Output dir          : /content/hate-speech-detection/models/xlmr_vihsd/checkpoints
   Num epochs          : 10
   Train batch size    : 16
   Eval batch size     : 32
   Learning rate       : 2e-05
   FP16                : True  ← MUST be True
   num_workers         : 0  ← MUST be 0
   eval_strategy       : IntervalStrategy.EPOCH
   save_total_limit    : 2
   metric_for_best     : eval_f1_macro
   Resume checkpoint   : None (fresh start)
[GPU | cell5_end] Allocated: 0.00GB | Reserved: 0.00GB | Free: 14.56GB / 14.56GB


In [ ]:
# ============================================================
# CELL 6 — TRAINING LOOP
# ============================================================
# ⏱️  ESTIMATED TIME : 45–60 minutes on T4 GPU (10 epochs)
# 💾  AUTO-SAVE      : Checkpoints saved to Drive after every epoch
# 🔄  RESUME         : If Colab crashes → re-run Cell 0→6
#                      (Cell 5 auto-detects the latest checkpoint)
#
# Input:
#   - MODEL_NAME      : from Cell 3
#   - training_args   : TrainingArguments from Cell 5
#   - train_dataset   : ViHSDDataset (train) from Cell 3
#   - val_dataset     : ViHSDDataset (val)   from Cell 3
#   - class_weights   : list[float] from Cell 2
#   - resume_checkpoint: str or None from Cell 5
#
# Output:
#   - model           : PhoBERT-v2 fine-tuned
#   - trainer         : Trained WeightedTrainer (best model loaded)
#   - train_result    : TrainOutput object with loss & step stats
# ============================================================
from transformers import EarlyStoppingCallback, AutoModelForSequenceClassification
from models.classifier import compute_metrics  # src/models/classifier.py

# --- 1. Model Initialization ---
print(f"Loading model: '{MODEL_NAME}'...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
)
print_gpu_memory("after_model_load")

# --- 2. Trainer Setup ---
# EarlyStoppingCallback: stop if val F1 does not improve for 3 consecutive epochs.
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=3,
    early_stopping_threshold=0.001,   # Minimum improvement to count as "better"
)

# Instantiate WeightedTrainer (defined in the previous cell)
trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping],
)

# Pre-training cleanup: free every byte of VRAM before the long run
clean_memory()
print_gpu_memory("pre_training")

print()
print("=" * 60)
print("🚀 STARTING TRAINING")
print()
if resume_checkpoint:
    print(f"   Mode     : RESUME from {os.path.basename(resume_checkpoint)}")
else:
    print(f"   Mode     : FRESH START (epoch 1 of {training_args.num_train_epochs})")
print(f"   Epochs   : {training_args.num_train_epochs} (early stop patience=3)")
print(f"   Batch    : {training_args.per_device_train_batch_size}")
print(f"   FP16     : {training_args.fp16}")
print(f"   Checkpts : {CHECKPOINTS_DIR}")
print()
print("   ⚠️  DO NOT interrupt this cell unless Colab disconnects.")
print("   ⚠️  If disconnected: re-run Cells 0→6, it will auto-resume.")
print("=" * 60)

# --- THE TRAINING CALL ---
train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)

# --- Post-training summary ---
print()
print("=" * 60)
print("✅ TRAINING COMPLETE")
print(f"   Global steps   : {train_result.global_step:,}")
print(f"   Training loss  : {train_result.training_loss:.4f}")
print(f"   Epochs trained : {train_result.global_step / (len(train_dataset) / (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)):.1f}")
print("=" * 60)
print_gpu_memory("post_training")

# Print full training history (loss + val metrics per epoch)
print()
print("📊 Training History (per epoch):")
print(f"{'Epoch':>6} | {'Train Loss':>12} | {'Val F1 (macro)':>14} | {'Val Loss':>10}")
print("-" * 55)

log_history = trainer.state.log_history
# Separate train_loss rows and eval rows
train_logs = [x for x in log_history if "loss" in x and "eval_loss" not in x]
eval_logs  = [x for x in log_history if "eval_f1_macro" in x]

for i, eval_log in enumerate(eval_logs):
    epoch      = eval_log.get("epoch", i + 1)
    val_f1     = eval_log.get("eval_f1_macro", "—")
    val_loss   = eval_log.get("eval_loss", "—")
    # Match training loss from the same epoch interval
    matching_train = [
        t["loss"] for t in train_logs
        if abs(t.get("epoch", 0) - epoch) < 1.05
    ]
    t_loss = round(matching_train[-1], 4) if matching_train else "—"
    print(f"{epoch:>6.1f} | {str(t_loss):>12} | {str(val_f1):>14} | {str(round(val_loss, 4) if isinstance(val_loss, float) else val_loss):>10}")

# Best model info
best_ckpt = trainer.state.best_model_checkpoint
best_metric = trainer.state.best_metric
print()
print(f"🏆 Best checkpoint : {os.path.basename(best_ckpt) if best_ckpt else 'last'}")
print(f"🏆 Best val F1     : {best_metric:.4f}" if isinstance(best_metric, float) else f"🏆 Best val F1: {best_metric}")


Loading model: 'vinai/phobert-base-v2'...


pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[GPU | after_model_load] Allocated: 0.00GB | Reserved: 0.00GB | Free: 14.56GB / 14.56GB


HfHubHTTPError: Client error '401 Unauthorized' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-6a06281d-21ea2d72338f1df32c21163a;e7995d50-1598-447f-a0cb-f2cc588061f7)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.

model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

In [ ]:
# ============================================================
# CELL 7 — SAVE FINAL MODEL & (OPTIONAL) PUSH TO HF HUB
# ============================================================
# Input:
#   - trainer   : Trained WeightedTrainer (best model already loaded)
#   - tokenizer : XLM-RoBERTa tokenizer from Cell 3
# Output:
#   - {PROJECT_ROOT}/models/xlmr_vihsd/final_hf/  (HuggingFace format)
#   - (Optional) Model pushed to HuggingFace Hub
#   - Old checkpoint folders deleted to free local space
# ============================================================
import shutil
import glob
import json
import os

FINAL_HF_DIR = resolve_path(cfg.models.final_hf_dir)
os.makedirs(FINAL_HF_DIR, exist_ok=True)

# --- Step 1: Save model + tokenizer to Local Storage in HuggingFace format ---
print(f"💾 Saving final model to Local Storage...")
print(f"   Path: {FINAL_HF_DIR}")

trainer.save_model(FINAL_HF_DIR)
tokenizer.save_pretrained(FINAL_HF_DIR)

# Verify the save
expected_files = ["config.json", "tokenizer.json", "tokenizer_config.json"]
for fname in expected_files:
    fpath = os.path.join(FINAL_HF_DIR, fname)
    if os.path.exists(fpath):
        print(f"   ✅ {fname}")
    else:
        print(f"   ⚠️  {fname} not found — may use safetensors format, check manually")

# Also save the best model metric for traceability
meta = {
    "backbone": str(cfg.model.backbone),
    "best_val_f1_macro": trainer.state.best_metric,
    "best_checkpoint": os.path.basename(trainer.state.best_model_checkpoint or ""),
    "class_weights": class_weights,
    "max_length": int(cfg.training.max_length),
    "num_epochs_trained": len([x for x in trainer.state.log_history if "eval_f1_macro" in x]),
}
meta_path = os.path.join(FINAL_HF_DIR, "training_meta.json")
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)
print(f"   ✅ training_meta.json (best F1={meta['best_val_f1_macro']})")
print(f"\n✅ Model saved to Local Storage: {FINAL_HF_DIR}")

# --- Step 2: (Optional) Push to HuggingFace Hub ---
HF_MODEL_ID = "thong7d/vihsd-xlmr-hate-speech"  # ← Fill this in! E.g. "thong7d/vihsd-xlmr-hate-speech"

if HF_MODEL_ID:
    print(f"\n📤 Pushing to HuggingFace Hub: {HF_MODEL_ID}")
    try:
        from huggingface_hub import get_token
        hf_token = get_token()
    except Exception:
        hf_token = None
        print("   ⚠️  HF_TOKEN not found. Attempting anonymous push...")

    try:
        trainer.model.push_to_hub(HF_MODEL_ID, token=hf_token)
        tokenizer.push_to_hub(HF_MODEL_ID, token=hf_token)
        print(f"   ✅ Pushed successfully!")
        print(f"   🌐 View at: https://huggingface.co/{HF_MODEL_ID}")

        print(f"\n   ⚠️  ACTION REQUIRED: Update configs/paths.yaml on your PC:")
        print(f'   hf_model_id: "{HF_MODEL_ID}"')
        print(f"   Then: git add configs/paths.yaml && git commit -m 'feat: add hf_model_id' && git push")

    except Exception as e:
        print(f"   ❌ Push failed: {e}")
        print(f"   Model is safely saved locally. Push manually later:")
else:
    print("\n   ℹ️  HF_MODEL_ID is empty → skipping Hub push.")
    print("   To push later: fill in HF_MODEL_ID and re-run this cell.")

# --- Step 3: Delete old checkpoint folders to free Local space ---
print(f"\n🗑️  Cleaning up intermediate checkpoints from Local Storage...")
checkpoint_pattern = os.path.join(CHECKPOINTS_DIR, "checkpoint-*")
deleted = 0
for ckpt_dir in glob.glob(checkpoint_pattern):
    shutil.rmtree(ckpt_dir, ignore_errors=True)
    print(f"   Deleted: {os.path.basename(ckpt_dir)}")
    deleted += 1

if deleted == 0:
    print("   No checkpoint folders to delete.")
else:
    print(f"   ✅ {deleted} checkpoint folder(s) removed → Local space freed.")

clean_memory()
print_gpu_memory("after_save")